# Notebook 08: Hybrid Quantum-Classical Networks with PyTorch

This notebook demonstrates integrating PennyLane QNodes into PyTorch neural network modules using `TorchLayer`.

---

## Learning Objectives
1. Convert QNodes into PyTorch-compatible layers.
2. Build a hybrid classical-quantum feedforward network.
3. Perform a forward pass with synthetic tensors.


---
## Real-World Applications & Modern Use Cases

Hybrid PyTorch quantum layers enable:
- **Enterprise AI Model Integration:** Embedding quantum layers (`qml.qnn.TorchLayer`) into standard production PyTorch deep learning architectures (e.g. ResNets, Vision Transformers).
- **Drug Discovery Property Prediction:** Combining classical graph neural networks (GNNs) with quantum layers to predict molecular binding affinities and toxicity.


In [1]:
import pennylane as qml
import torch
import torch.nn as nn

dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=[0, 1])
    qml.RY(weights[0], wires=0)
    qml.RY(weights[1], wires=1)
    qml.CNOT(wires=[0, 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(2)]

qlayer = qml.qnn.TorchLayer(qnode, {"weights": 2})

class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Linear(4, 2)
        self.quantum = qlayer
        self.post = nn.Linear(2, 1)
        
    def forward(self, x):
        x = torch.relu(self.pre(x))
        x = self.quantum(x)
        return torch.sigmoid(self.post(x))

model = HybridModel()
dummy_data = torch.randn(2, 4)
output = model(dummy_data)

print("Model output tensor shape:", output.shape)
print("Sample Predictions:", output.detach().numpy())


Model output tensor shape: torch.Size([2, 1])
Sample Predictions: [[0.7231097 ]
 [0.74795026]]
